# Mi evaluación de metadata y tarifas pendientes
Consulté los snapshots existentes 2025-10-28 y 2026-09-29 mediante Genie.
Estos seis bloques solo leen; no implementan ni publican contratos Silver.
Conservo el SQL completo ejecutado, no las versiones abreviadas del resumen de Genie.
No es una etapa obligatoria del Job ni necesito reejecutarlo antes de cada carga.
Los valores y los nombres de entorno son contexto de esta revisión, no configuración de producción.


## Bloque 1 — Presencia auditada de archivos por snapshot


In [ ]:
%sql
WITH snapshots AS (
  SELECT '2025-10-28' AS version, '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' AS hash
  UNION ALL
  SELECT '2026-09-29', '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
)
SELECT 
  s.version AS snapshot_label,
  MAX(CASE WHEN obs.`source_file_name` = 'feed_info.txt' THEN 1 ELSE 0 END) AS feed_info_file_present,
  MAX(CASE WHEN obs.`source_file_name` = 'fare_attributes.txt' THEN 1 ELSE 0 END) AS fare_attributes_file_present,
  MAX(CASE WHEN obs.`source_file_name` = 'fare_rules.txt' THEN 1 ELSE 0 END) AS fare_rules_file_present
FROM snapshots s
LEFT JOIN `workspace`.`gtfs_bogota`.`audit_gtfs_source_schema_observations` obs
  ON s.version = obs.`source_feed_version`
  AND s.hash = obs.`source_sha256`
  AND obs.`source_file_name` IN ('feed_info.txt', 'fare_attributes.txt', 'fare_rules.txt')
GROUP BY s.version
ORDER BY s.version


## Bloque 2 — Claves, importes y agencias de fare_attributes


In [ ]:
%sql
WITH snapshots AS (
  SELECT '2025-10-28' AS version, '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' AS hash
  UNION ALL
  SELECT '2026-09-29', '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
),
raw_data AS (
  SELECT 
    f.*,
    s.version AS snapshot_label
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_fare_attributes` f
  INNER JOIN snapshots s 
    ON f.`source_feed_version` = s.version 
    AND f.`source_sha256` = s.hash
),
profile AS (
  SELECT
    snapshot_label,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT `fare_id`) AS distinct_fare_ids,
    -- Nulos/vacíos en campos clave
    SUM(CASE WHEN `fare_id` IS NULL OR TRIM(`fare_id`) = '' THEN 1 ELSE 0 END) AS fare_id_null_empty,
    SUM(CASE WHEN `price` IS NULL OR TRIM(`price`) = '' THEN 1 ELSE 0 END) AS price_null_empty,
    SUM(CASE WHEN `currency_type` IS NULL OR TRIM(`currency_type`) = '' THEN 1 ELSE 0 END) AS currency_null_empty,
    SUM(CASE WHEN `payment_method` IS NULL OR TRIM(`payment_method`) = '' THEN 1 ELSE 0 END) AS payment_method_null_empty,
    -- IDs repetidos
    SUM(CASE WHEN fare_id_count > 1 THEN 1 ELSE 0 END) AS repeated_fare_ids,
    -- Validación de price
    SUM(CASE WHEN `price` IS NOT NULL AND TRIM(`price`) != '' 
             AND NOT `price` RLIKE '^[0-9]+(\\.[0-9]+)?$' THEN 1 ELSE 0 END) AS price_non_numeric,
    SUM(CASE WHEN TRY_CAST(`price` AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS price_negative,
    -- Min/Max precio
    MIN(TRY_CAST(`price` AS DOUBLE)) AS min_price,
    MAX(TRY_CAST(`price` AS DOUBLE)) AS max_price,
    -- Códigos de moneda
    COUNT(DISTINCT `currency_type`) AS distinct_currencies,
    MAX(`currency_type`) AS sample_currency,
    -- Valores de payment_method y transfers
    COUNT(DISTINCT `payment_method`) AS distinct_payment_methods,
    COUNT(DISTINCT `transfers`) AS distinct_transfers_values,
    MAX(`payment_method`) AS sample_payment_method,
    MAX(`transfers`) AS sample_transfers,
    -- Agency_id no blanco
    SUM(CASE WHEN `agency_id` IS NOT NULL AND TRIM(`agency_id`) != '' THEN 1 ELSE 0 END) AS rows_with_agency_id
  FROM (
    SELECT 
      *,
      COUNT(*) OVER (PARTITION BY snapshot_label, `fare_id`) AS fare_id_count
    FROM raw_data
  ) sub
  GROUP BY snapshot_label
),
agency_check AS (
  SELECT
    fa.snapshot_label,
    COUNT(DISTINCT fa.`agency_id`) AS distinct_agency_ids_referenced,
    COUNT(DISTINCT ag.`agency_id`) AS agency_ids_found_in_silver,
    SUM(CASE WHEN ag.`agency_id` IS NULL THEN 1 ELSE 0 END) AS agency_refs_not_found
  FROM raw_data fa
  LEFT JOIN `workspace`.`gtfs_bogota`.`silver_gtfs_agency` ag
    ON fa.`source_feed_version` = ag.`source_feed_version`
    AND fa.`source_sha256` = ag.`source_sha256`
    AND fa.`agency_id` = ag.`agency_id`
  WHERE fa.`agency_id` IS NOT NULL AND TRIM(fa.`agency_id`) != ''
  GROUP BY fa.snapshot_label
)
SELECT 
  p.*,
  COALESCE(ac.distinct_agency_ids_referenced, 0) AS distinct_agency_ids_referenced,
  COALESCE(ac.agency_ids_found_in_silver, 0) AS agency_ids_found_in_silver,
  COALESCE(ac.agency_refs_not_found, 0) AS agency_refs_not_found
FROM profile p
LEFT JOIN agency_check ac ON p.snapshot_label = ac.snapshot_label
ORDER BY p.snapshot_label


## Bloque 3 — Reglas, tarifas y rutas de fare_rules


In [ ]:
%sql
WITH snapshots AS (
  SELECT '2025-10-28' AS version, '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' AS hash
  UNION ALL
  SELECT '2026-09-29', '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
),
raw_data AS (
  SELECT 
    fr.*,
    s.version AS snapshot_label
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_fare_rules` fr
  INNER JOIN snapshots s 
    ON fr.`source_feed_version` = s.version 
    AND fr.`source_sha256` = s.hash
),
profile AS (
  SELECT
    snapshot_label,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT_WS('|',
      COALESCE(`fare_id`, 'NULL'),
      COALESCE(`route_id`, 'NULL'),
      COALESCE(`origin_id`, 'NULL'),
      COALESCE(`destination_id`, 'NULL'),
      COALESCE(`contains_id`, 'NULL')
    )) AS distinct_content_rows,
    -- Nulos/vacíos de fare_id
    SUM(CASE WHEN `fare_id` IS NULL OR TRIM(`fare_id`) = '' THEN 1 ELSE 0 END) AS fare_id_null_empty,
    -- Campos de regla publicados
    SUM(CASE WHEN `route_id` IS NOT NULL AND TRIM(`route_id`) != '' THEN 1 ELSE 0 END) AS rows_with_route_id,
    SUM(CASE WHEN `origin_id` IS NOT NULL AND TRIM(`origin_id`) != '' THEN 1 ELSE 0 END) AS rows_with_origin_id,
    SUM(CASE WHEN `destination_id` IS NOT NULL AND TRIM(`destination_id`) != '' THEN 1 ELSE 0 END) AS rows_with_destination_id,
    SUM(CASE WHEN `contains_id` IS NOT NULL AND TRIM(`contains_id`) != '' THEN 1 ELSE 0 END) AS rows_with_contains_id,
    COUNT(DISTINCT `fare_id`) AS distinct_fare_ids,
    COUNT(DISTINCT `route_id`) AS distinct_route_ids
  FROM raw_data
  GROUP BY snapshot_label
),
fare_id_check AS (
  SELECT
    fr.snapshot_label,
    COUNT(DISTINCT fr.`fare_id`) AS distinct_fare_ids_in_rules,
    COUNT(DISTINCT fa.`fare_id`) AS fare_ids_found_in_attributes,
    SUM(CASE WHEN fa.`fare_id` IS NULL THEN 1 ELSE 0 END) AS fare_refs_not_found
  FROM raw_data fr
  LEFT JOIN `workspace`.`gtfs_bogota`.`bronze_gtfs_fare_attributes` fa
    ON fr.`source_feed_version` = fa.`source_feed_version`
    AND fr.`source_sha256` = fa.`source_sha256`
    AND fr.`fare_id` = fa.`fare_id`
  WHERE fr.`fare_id` IS NOT NULL AND TRIM(fr.`fare_id`) != ''
  GROUP BY fr.snapshot_label
),
route_check AS (
  SELECT
    fr.snapshot_label,
    COUNT(DISTINCT fr.`route_id`) AS distinct_route_ids_referenced,
    COUNT(DISTINCT rt.`route_id`) AS route_ids_found_in_silver,
    SUM(CASE WHEN rt.`route_id` IS NULL THEN 1 ELSE 0 END) AS route_refs_not_found
  FROM raw_data fr
  LEFT JOIN `workspace`.`gtfs_bogota`.`silver_gtfs_routes` rt
    ON fr.`source_feed_version` = rt.`source_feed_version`
    AND fr.`source_sha256` = rt.`source_sha256`
    AND fr.`route_id` = rt.`route_id`
  WHERE fr.`route_id` IS NOT NULL AND TRIM(fr.`route_id`) != ''
  GROUP BY fr.snapshot_label
)
SELECT 
  p.*,
  COALESCE(fc.distinct_fare_ids_in_rules, 0) AS distinct_fare_ids_in_rules,
  COALESCE(fc.fare_ids_found_in_attributes, 0) AS fare_ids_found_in_attributes,
  COALESCE(fc.fare_refs_not_found, 0) AS fare_refs_not_found,
  COALESCE(rc.distinct_route_ids_referenced, 0) AS distinct_route_ids_referenced,
  COALESCE(rc.route_ids_found_in_silver, 0) AS route_ids_found_in_silver,
  COALESCE(rc.route_refs_not_found, 0) AS route_refs_not_found
FROM profile p
LEFT JOIN fare_id_check fc ON p.snapshot_label = fc.snapshot_label
LEFT JOIN route_check rc ON p.snapshot_label = rc.snapshot_label
ORDER BY p.snapshot_label


## Bloque 4 — Metadata y fechas declaradas de feed_info


In [ ]:
%sql
WITH snapshots AS (
  SELECT '2025-10-28' AS version, '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' AS hash
  UNION ALL
  SELECT '2026-09-29', '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
),
raw_data AS (
  SELECT 
    f.*,
    s.version AS snapshot_label
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_feed_info` f
  INNER JOIN snapshots s 
    ON f.`source_feed_version` = s.version 
    AND f.`source_sha256` = s.hash
),
profile AS (
  SELECT
    snapshot_label,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT_WS('|', 
      COALESCE(`feed_publisher_name`, 'NULL'),
      COALESCE(`feed_lang`, 'NULL'),
      COALESCE(`feed_version`, 'NULL'),
      COALESCE(`feed_start_date`, 'NULL'),
      COALESCE(`feed_end_date`, 'NULL'),
      COALESCE(`feed_publisher_url`, 'NULL'),
      COALESCE(`feed_contact_email`, 'NULL')
    )) AS distinct_content_rows,
    -- Campos publicados
    COUNT(DISTINCT `feed_publisher_name`) AS distinct_publishers,
    COUNT(DISTINCT `feed_lang`) AS distinct_langs,
    COUNT(DISTINCT `feed_version`) AS distinct_versions,
    COUNT(DISTINCT `feed_start_date`) AS distinct_start_dates,
    COUNT(DISTINCT `feed_end_date`) AS distinct_end_dates,
    -- Nulos/vacíos en campos requeridos
    SUM(CASE WHEN `feed_publisher_name` IS NULL OR TRIM(`feed_publisher_name`) = '' THEN 1 ELSE 0 END) AS publisher_name_null_empty,
    SUM(CASE WHEN `feed_publisher_url` IS NULL OR TRIM(`feed_publisher_url`) = '' THEN 1 ELSE 0 END) AS publisher_url_null_empty,
    SUM(CASE WHEN `feed_lang` IS NULL OR TRIM(`feed_lang`) = '' THEN 1 ELSE 0 END) AS lang_null_empty,
    -- Valores publicados (primeros encontrados)
    MAX(`feed_publisher_name`) AS sample_publisher,
    MAX(`feed_lang`) AS sample_lang,
    MAX(`feed_version`) AS sample_version,
    MAX(`feed_start_date`) AS sample_start_date,
    MAX(`feed_end_date`) AS sample_end_date
  FROM raw_data
  GROUP BY snapshot_label
),
date_validation AS (
  SELECT
    snapshot_label,
    `feed_start_date`,
    `feed_end_date`,
    CASE 
      WHEN LENGTH(`feed_start_date`) = 8 AND `feed_start_date` RLIKE '^[0-9]{8}$'
      THEN TRY_CAST(TO_DATE(`feed_start_date`, 'yyyyMMdd') AS STRING)
      ELSE 'INVALID'
    END AS start_date_converted,
    CASE 
      WHEN LENGTH(`feed_end_date`) = 8 AND `feed_end_date` RLIKE '^[0-9]{8}$'
      THEN TRY_CAST(TO_DATE(`feed_end_date`, 'yyyyMMdd') AS STRING)
      ELSE 'INVALID'
    END AS end_date_converted,
    CASE 
      WHEN LENGTH(`feed_start_date`) = 8 AND LENGTH(`feed_end_date`) = 8
           AND `feed_start_date` RLIKE '^[0-9]{8}$' AND `feed_end_date` RLIKE '^[0-9]{8}$'
           AND TO_DATE(`feed_start_date`, 'yyyyMMdd') > TO_DATE(`feed_end_date`, 'yyyyMMdd')
      THEN 1 ELSE 0
    END AS inverted_dates
  FROM raw_data
  WHERE `feed_start_date` IS NOT NULL OR `feed_end_date` IS NOT NULL
)
SELECT 
  p.*,
  COALESCE(SUM(dv.inverted_dates), 0) AS inverted_date_pairs,
  MAX(dv.start_date_converted) AS sample_start_converted,
  MAX(dv.end_date_converted) AS sample_end_converted
FROM profile p
LEFT JOIN date_validation dv ON p.snapshot_label = dv.snapshot_label
GROUP BY 
  p.snapshot_label, p.total_rows, p.distinct_content_rows,
  p.distinct_publishers, p.distinct_langs, p.distinct_versions,
  p.distinct_start_dates, p.distinct_end_dates,
  p.publisher_name_null_empty, p.publisher_url_null_empty, p.lang_null_empty,
  p.sample_publisher, p.sample_lang, p.sample_version,
  p.sample_start_date, p.sample_end_date
ORDER BY p.snapshot_label


## Bloque 5 — Referencias de zonas de las reglas de 2025


In [ ]:
%sql
with zones as (
  select distinct source_feed_version, source_sha256, zone_id
  from workspace.gtfs_bogota.silver_gtfs_stops
  where source_feed_version = '2025-10-28'
    and source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
    and zone_id is not null and trim(zone_id) <> ''
), rules as (
  select source_feed_version, source_sha256, origin_id, destination_id, contains_id
  from workspace.gtfs_bogota.bronze_gtfs_fare_rules
  where source_feed_version = '2025-10-28'
    and source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
)
select r.source_feed_version, count(*) as rule_rows,
       count(distinct r.origin_id) as distinct_origin_ids,
       sum(case when r.origin_id is not null and trim(r.origin_id) <> '' and o.zone_id is null then 1 else 0 end) as origin_refs_not_found,
       sum(case when r.destination_id is not null and trim(r.destination_id) <> '' and d.zone_id is null then 1 else 0 end) as destination_refs_not_found,
       sum(case when r.contains_id is not null and trim(r.contains_id) <> '' and c.zone_id is null then 1 else 0 end) as contains_refs_not_found
from rules r
left join zones o on r.source_feed_version=o.source_feed_version and r.source_sha256=o.source_sha256 and r.origin_id=o.zone_id
left join zones d on r.source_feed_version=d.source_feed_version and r.source_sha256=d.source_sha256 and r.destination_id=d.zone_id
left join zones c on r.source_feed_version=c.source_feed_version and r.source_sha256=c.source_sha256 and r.contains_id=c.zone_id
group by r.source_feed_version


## Bloque 6 — Detalle tarifario acotado y agencia del mismo snapshot


In [ ]:
%sql
with snapshots as (
 select '2025-10-28' as version, '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' as hash
 union all select '2026-09-29', '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
), agencies as (
 select distinct source_feed_version, source_sha256, agency_id, agency_name
 from workspace.gtfs_bogota.silver_gtfs_agency
)
select f.source_feed_version, f.fare_id, f.price, f.currency_type,
       f.agency_id, a.agency_name, f.payment_method, f.transfers, f.transfer_duration
from workspace.gtfs_bogota.bronze_gtfs_fare_attributes f
join snapshots s on f.source_feed_version=s.version and f.source_sha256=s.hash
left join agencies a on f.source_feed_version=a.source_feed_version and f.source_sha256=a.source_sha256 and f.agency_id=a.agency_id
order by f.source_feed_version, f.fare_id
limit 20


## Límites
La revisión comprueba controles seleccionados de los datos presentes, no una conformidad GTFS completa.
Los SQL de fechas y duplicados describen los valores actuales; no sustituyen validadores de producción.
Destination_id y contains_id publicados sin contenido no son columnas ausentes.
Precio cero no demuestra por sí solo gratuidad ni subsidio.
Transfers vacío tiene semántica propia del estándar; no lo sustituyo automáticamente por cero.
La tarifa de 2026 está asociada a TransMiCable; no la extiendo automáticamente a toda la red.
No infiero ingresos, ocupación ni tarifas vigentes a partir del snapshot.
